#<font color="Green">**Notebook Purpose**</font>

This notebook constructs the comorbidity table used in downstream analyses. Starting from the raw comorbidity diagnoses and the finalized study cohort, it produces a patient-level CSV file with binary indicators for key baseline comorbid conditions.

More specifically, this notebook:

1. **Restricts comorbidities to the analytic cohort:**  
   Filters the comorbidity records to patients included in the final clustering cohort.

2. **Applies temporal restrictions:**  
   Retains only comorbidity diagnoses that occur on or before the specified baseline date (e.g., 2019-01-01), so that comorbidities reflect pre-treatment or baseline status.

3. **Derives comorbidity indicators:**  
   Creates patient-level flag variables (e.g., indicators for heart failure, chronic kidney disease, and relevant combinations such as concurrent HF and CKD) based on diagnosis codes.

4. **Outputs a cleaned comorbidity file:**  
   Saves a consolidated `patient_comorbidities.csv` file with one row per patient and the derived comorbidity indicators for use in later analysis notebooks.

---

###<font color ="Red"> Required Data </font>

To run this notebook, the following input files are required in the working directory:

1. **`clustering_patient_ids.csv`**  
   - Contains the list of patient IDs included in the final analytic cohort.  
   - At minimum, must include a `patient_id` column.

2. **`comorbidities.csv`**  
   - Long-format comorbidity diagnoses table returned by the TriNetX query (see 'Queries' folder).  
   - Must include, at minimum:  
     - `patient_id` – unique patient identifier  
     - `date` – diagnosis date  
     - `code` – diagnosis code (e.g., ICD-10)  
   - Additional columns (e.g., code system) may be present but are not strictly required for the derivation of comorbidity indicators.

The notebook assumes that diagnosis codes in `comorbidities.csv` follow the ICD-10 convention used in the TriNetX data dictionary and that all patients in `clustering_patient_ids.csv` are part of the study’s final cohort.


In [ ]:
import pandas as pd
import numpy as np
import ast

##<font color="black">**Read In Data**</font>

In [ ]:
final_patients = pd.read_csv('/content/clustering_patient_ids.csv')
final_patients.head()

In [ ]:
patient_comorbidities = pd.read_csv('/content/comorbidities.csv')
patient_comorbidities.head()

##<font color="black">**Remove Duplicates**</font>

In [ ]:
print('Duplicate entries:', patient_comorbidities.duplicated().sum())
patient_comorbidities = patient_comorbidities.drop_duplicates()

In [ ]:
patient_comorbidities.head()

##<font color = "Black"> **Preliminary Analysis & Cleaning** </font>

In [ ]:
print('Number of patients who have comorbidities: ', len(patient_comorbidities['patient_id'].unique().tolist()))

Need ensure that patients represented in patient_comorbidities are the correct patients

In [ ]:
final_patients['patient_id'] = final_patients['patient_id'].astype(str)
patient_comorbidities['patient_id'] = patient_comorbidities['patient_id'].astype(str)

# keep only rows whose patient_id exists in the reference list
patient_comorbidities = (
    patient_comorbidities
    .merge(final_patients[['patient_id']].drop_duplicates(), on='patient_id', how='inner')
)

print('Number of patients who have comorbidities: ', len(patient_comorbidities['patient_id'].unique().tolist()))

In [ ]:
print('Number of patients who have T2D with CKD (E11.22): ', len(patient_comorbidities[patient_comorbidities['code'] == 'E11.22']['patient_id'].unique().tolist()))

In [ ]:
print('Max diagnosis date: ', patient_comorbidities['date'].max())
print('Min diagnosis date: ', patient_comorbidities['date'].min())

I don't want any comorbidity data from after 2024-12-31

In [ ]:
patient_comorbidities = patient_comorbidities[patient_comorbidities['date'] <= '2024-12-31']
print('Max diagnosis date: ', patient_comorbidities['date'].max())
print('Number of patients who have comorbidities: ', len(patient_comorbidities['patient_id'].unique().tolist()))

In [ ]:
patients_with_comorbidities_pre_2019 = patient_comorbidities[patient_comorbidities['date'] <= '2019-01-01']
print('Number of patients who have a comorbidity diagnosis before 2019: ', len(patients_with_comorbidities_pre_2019['patient_id'].unique().tolist()))

##<font color = "Black"> **Creating and Exporting Final Table** </font>

In [ ]:
patient_comorbidities[patient_comorbidities['patient_id'] == '2A4qC']

Our goal is to produce an updated version of this table that includes additional columns identifying whether each diagnosis code reflects heart failure (HF), chronic kidney disease (CKD), or both conditions

Creating a column called HF.

In [ ]:
HF_codes = set()
for _, row in patient_comorbidities.iterrows():
  code = row['code']
  if code.startswith('I50') or code == 'I11.0':
    HF_codes.add(code)

print(HF_codes)

In [ ]:
HF_codes.add('I50.81')
HF_codes.add('I50.8')
HF_codes.add('I50.3')
print(HF_codes)

In [ ]:
mask_HF = patient_comorbidities['code'].isin(HF_codes)
patient_comorbidities['HF'] = mask_HF
patient_comorbidities.head()

Creating a column called CKD.

In [ ]:
CKD_codes = set()
for _, row in patient_comorbidities.iterrows():
  code = row['code']
  if code.startswith('I12') or code.startswith('N18') or code == 'E11.22':
    CKD_codes.add(code)

print(CKD_codes)

In [ ]:
CKD_codes.add('I12')

In [ ]:
mask_CKD = patient_comorbidities['code'].isin(CKD_codes)
patient_comorbidities['CKD'] = mask_CKD
patient_comorbidities.head()

Dealing with combination of HF and CKD

In [ ]:
codes = patient_comorbidities['code'].astype(str).str.strip().str.upper()
mask = codes.isin({'I13.0', 'I13.2'})  # or .str.startswith(('I13.0','I13.2'))

patient_comorbidities.loc[mask, ['CKD', 'HF']] = True

In [ ]:
patient_comorbidities.head()

Exporting the final table

In [ ]:
patient_comorbidities.to_csv('patient_comorbidities.csv', index=False)